# CNS3102 Assignment 2: Machine Learning with Python
**Group: arif&ilham**

**Datasets:**
1. Predict Students' Dropout and Academic Success (UCI #697) — classification
2. Higher Education Students Performance Evaluation (UCI #856) — classification

This notebook covers dataset loading, EDA, preprocessing, model training (Logistic Regression, K-Nearest Neighbours, Decision Tree), and performance evaluation (accuracy, training/testing time, confusion matrices) for both datasets.


In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

sns.set_theme(style="whitegrid")
%matplotlib inline


: 

## Dataset 1: Predict Students' Dropout and Academic Success

- **Source:** UCI Machine Learning Repository
- **Link:** https://archive.ics.uci.edu/dataset/697/predict+students+dropout+and+academic+success
- **Objective:** Predict whether a student will Dropout, remain Enrolled, or Graduate, based on academic, demographic, and socioeconomic features.


In [ ]:
# In Colab: upload dropout.csv via the Files panel first
d1 = pd.read_csv("dropout.csv")
print(d1.shape)
d1.head()


In [ ]:
d1.describe()

In [ ]:
print("Missing values:", d1.isna().sum().sum())
print("Duplicate rows:", d1.duplicated().sum())
d1["Target"].value_counts()


In [ ]:
order = ["Dropout", "Enrolled", "Graduate"]
plt.figure(figsize=(6,4))
sns.countplot(data=d1, x="Target", order=order, hue="Target", palette="Blues_d", legend=False)
plt.title("Distribution of Student Academic Outcomes")
plt.xlabel("Student Outcome"); plt.ylabel("Number of Students")
plt.show()


In [ ]:
plt.figure(figsize=(7,4.5))
sns.histplot(data=d1, x="Age at enrollment", hue="Target", hue_order=order, kde=True, palette="muted")
plt.title("Age at Enrollment by Student Outcome")
plt.show()


In [ ]:
plt.figure(figsize=(6,4.5))
sns.boxplot(data=d1, x="Target", y="Curricular units 2nd sem (grade)", order=order, hue="Target", palette="Blues", legend=False)
plt.title("Second-Semester Grade by Student Outcome")
plt.show()


In [ ]:
corr_cols = ["Previous qualification", "Age at enrollment",
             "Curricular units 1st sem (approved)", "Curricular units 1st sem (grade)",
             "Curricular units 2nd sem (approved)", "Curricular units 2nd sem (grade)",
             "Unemployment rate", "Inflation rate", "GDP"]
plt.figure(figsize=(9,7))
sns.heatmap(d1[corr_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm", annot_kws={"size":7})
plt.title("Correlation Heatmap of Selected Numerical Features")
plt.show()


### Preprocessing
No missing values or duplicates were found. All features are already numerically encoded. Data is split 80/20 (stratified) and standardized.

In [ ]:
X1 = d1.drop(columns=["Target"])
y1 = d1["Target"]

X1_train, X1_test, y1_train, y1_test = train_test_split(X1, y1, test_size=0.2, random_state=42, stratify=y1)
scaler1 = StandardScaler()
X1_train_s = scaler1.fit_transform(X1_train)
X1_test_s = scaler1.transform(X1_test)
print(X1_train.shape, X1_test.shape)


In [ ]:
models1 = {
    "Logistic Regression": LogisticRegression(max_iter=2000, random_state=42),
    "K-Nearest Neighbours": KNeighborsClassifier(n_neighbors=7),
    "Decision Tree": DecisionTreeClassifier(random_state=42, max_depth=8),
}
labels1 = order
results1 = []
cms1 = {}
for name, model in models1.items():
    t0 = time.time(); model.fit(X1_train_s, y1_train); train_t = time.time()-t0
    t0 = time.time(); pred = model.predict(X1_test_s); test_t = time.time()-t0
    acc = accuracy_score(y1_test, pred)
    results1.append({"Model": name, "Accuracy (%)": round(acc*100,2),
                      "Training Time (s)": round(train_t,4), "Testing Time (s)": round(test_t,4)})
    cms1[name] = confusion_matrix(y1_test, pred, labels=labels1)
    print(f"--- {name} ---")
    print(classification_report(y1_test, pred))

results1_df = pd.DataFrame(results1)
results1_df


In [ ]:
plt.figure(figsize=(6,4))
sns.barplot(data=results1_df, x="Model", y="Accuracy (%)", hue="Model", palette="viridis", legend=False)
plt.ylim(0,100); plt.title("Dataset 1: Model Accuracy Comparison")
plt.xticks(rotation=15); plt.tight_layout(); plt.show()


In [ ]:
for name, cm in cms1.items():
    plt.figure(figsize=(5,4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=labels1, yticklabels=labels1)
    plt.title(f"{name} Confusion Matrix")
    plt.xlabel("Predicted"); plt.ylabel("Actual")
    plt.show()


---
## Dataset 2: Higher Education Students Performance Evaluation

- **Source:** UCI Machine Learning Repository
- **Link:** https://archive.ics.uci.edu/dataset/856/higher+education+students+performance+evaluation
- **Objective:** Predict a student's overall academic performance category (GRADE, 8 classes) from personal, family, and study-habit attributes.


In [ ]:
# Note: this dataset is semicolon-delimited
d2 = pd.read_csv("perf.csv", sep=";")
print(d2.shape)
d2.head()


In [ ]:
print("Missing values:", d2.isna().sum().sum())
print("Duplicate rows:", d2.duplicated().sum())
d2["GRADE"].value_counts().sort_index()


In [ ]:
# Column "17" corresponds to "Weekly study hours" in the original questionnaire
d2 = d2.rename(columns={"17": "Weekly study hours"})

plt.figure(figsize=(6,4))
sns.countplot(data=d2, x="GRADE", hue="GRADE", palette="Blues_d", legend=False)
plt.title("Distribution of Student Performance (OUTPUT Grade)")
plt.xlabel("Grade Category"); plt.ylabel("Number of Students")
plt.show()


In [ ]:
plt.figure(figsize=(6,4))
sns.histplot(d2["Weekly study hours"], bins=5, color="#4C72B0", edgecolor="white")
plt.title("Distribution of Weekly Study Hours")
plt.show()


In [ ]:
plt.figure(figsize=(6,4.5))
sns.boxplot(data=d2, x="GRADE", y="Weekly study hours", hue="GRADE", palette="Blues", legend=False)
plt.title("Weekly Study Hours by Grade")
plt.show()


In [ ]:
num_cols2 = d2.drop(columns=["STUDENT ID"]).columns
plt.figure(figsize=(11,9))
sns.heatmap(d2[num_cols2].corr(), cmap="coolwarm")
plt.title("Correlation Heatmap")
plt.xticks(fontsize=6, rotation=90); plt.yticks(fontsize=6)
plt.tight_layout(); plt.show()


### Preprocessing
The Student ID identifier column is dropped. Data is split 80/20 (stratified) and standardized.

In [ ]:
X2 = d2.drop(columns=["STUDENT ID","GRADE"])
y2 = d2["GRADE"]

X2_train, X2_test, y2_train, y2_test = train_test_split(X2, y2, test_size=0.2, random_state=42, stratify=y2)
scaler2 = StandardScaler()
X2_train_s = scaler2.fit_transform(X2_train)
X2_test_s = scaler2.transform(X2_test)
print(X2_train.shape, X2_test.shape)


In [ ]:
models2 = {
    "Logistic Regression": LogisticRegression(max_iter=2000, random_state=42),
    "K-Nearest Neighbours": KNeighborsClassifier(n_neighbors=5),
    "Decision Tree": DecisionTreeClassifier(random_state=42, max_depth=6),
}
labels2 = sorted(y2.unique())
results2 = []
cms2 = {}
for name, model in models2.items():
    t0 = time.time(); model.fit(X2_train_s, y2_train); train_t = time.time()-t0
    t0 = time.time(); pred = model.predict(X2_test_s); test_t = time.time()-t0
    acc = accuracy_score(y2_test, pred)
    results2.append({"Model": name, "Accuracy (%)": round(acc*100,2),
                      "Training Time (s)": round(train_t,4), "Testing Time (s)": round(test_t,4)})
    cms2[name] = confusion_matrix(y2_test, pred, labels=labels2)
    print(f"--- {name} ---")
    print(classification_report(y2_test, pred, zero_division=0))

results2_df = pd.DataFrame(results2)
results2_df


In [ ]:
plt.figure(figsize=(6,4))
sns.barplot(data=results2_df, x="Model", y="Accuracy (%)", hue="Model", palette="viridis", legend=False)
plt.ylim(0,100); plt.title("Dataset 2: Model Accuracy Comparison")
plt.xticks(rotation=15); plt.tight_layout(); plt.show()


In [ ]:
for name, cm in cms2.items():
    plt.figure(figsize=(5.5,4.5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=labels2, yticklabels=labels2)
    plt.title(f"{name} Confusion Matrix (Dataset 2)")
    plt.xlabel("Predicted"); plt.ylabel("Actual")
    plt.show()


---
## Summary

| Dataset | Best Model(s) | Accuracy |
|---|---|---|
| Dataset 1 (Dropout) | Logistic Regression | 75.82% |
| Dataset 2 (Performance) | Logistic Regression / Decision Tree (tied) | 27.59% |

See the accompanying report for the full discussion, methodology, and conclusions.
